# LightGBM v4 — 47 features, no leakage, heldout RMSE on 3 test wells

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# ── Paths ──────────────────────────────────────────────────────────────────────
def find_kaggle_raw_dir():
    input_root = Path('/kaggle/input')
    candidates = [input_root / 'rogii-wellbore-geology-prediction']
    candidates.extend(sorted(path.parent for path in input_root.rglob('sample_submission.csv')))
    candidates.extend(sorted(input_root.glob('*')))
    for candidate in candidates:
        if (
            candidate.exists()
            and (candidate / 'train').exists()
            and (candidate / 'test').exists()
            and (candidate / 'sample_submission.csv').exists()
        ):
            return candidate
    available = sorted(str(p) for p in input_root.glob('*')) if input_root.exists() else []
    raise FileNotFoundError(f'Could not find Kaggle raw data. Available: {available}')

if Path('/kaggle/working').exists():
    RAW_DIR = find_kaggle_raw_dir()
    OUTPUT_PATH = Path('/kaggle/working/submission.csv')
else:
    RAW_DIR = Path('../data/raw') if Path('../data/raw').exists() else Path('data/raw')
    OUTPUT_PATH = Path('../submissions/lightgbm_v4_alltrain.csv') if Path('../submissions').exists() else Path('submissions/lightgbm_v4_alltrain.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f'RAW_DIR={RAW_DIR.resolve()}')
print(f'OUTPUT_PATH={OUTPUT_PATH.resolve()}')

# The 3 Kaggle test wells also exist in raw/train/ with full TVT.
# We EXCLUDE them from training to avoid leakage,
# then use them as a held-out set to estimate Kaggle RMSE.
TEST_WELL_IDS = {'000d7d20', '00bbac68', '00e12e8b'}

RANDOM_STATE = 42
MAX_TRAIN_ROWS = 9_999_999  # use all available rows

FEATURE_COLUMNS = [
    'row_index', 'n_rows', 'row_from_ps', 'frac_from_ps',
    'MD', 'md_from_ps',
    'Z', 'x_from_ps', 'y_from_ps', 'z_from_ps', 'xy_dist_from_ps',
    'gr', 'gr_was_missing', 'gr_from_ps',
    'gr_roll_mean_11', 'gr_roll_mean_51', 'gr_roll_std_51',
    'gr_delta_1', 'gr_delta_10',
    'last_tvt_input', 'first_tvt_input', 'tvt_input_range',
    'tvt_slope_last_25', 'tvt_slope_last_100',
    'baseline_tvt',
    'typewell_tvt_min', 'typewell_tvt_max',
    'typewell_gr_at_baseline_tvt', 'gr_minus_typewell_gr_at_baseline',
    # v4: GR lags/leads
    'gr_lag_1', 'gr_lag_2', 'gr_lag_3', 'gr_lag_4', 'gr_lag_5',
    'gr_lead_1', 'gr_lead_2', 'gr_lead_3', 'gr_lead_4', 'gr_lead_5',
    # v4: local window stats
    'gr_local_mean_5', 'gr_local_std_5',
    # v4: GR anomaly
    'gr_anomaly',
    # v4: window-based template matching
    'nearest_typewell_tvt_by_gr_window', 'nearest_typewell_window_mse',
    'typewell_gr_at_window_match', 'gr_minus_typewell_gr_at_window_match',
    # v4: estimator discrepancy
    'typewell_tvt_vs_baseline',
]
META_COLUMNS = ['well_id', 'row_index']
TARGET_COLUMN = 'target_tvt'

In [ ]:
# ── Feature engineering helpers ────────────────────────────────────────────────

def interpolate_with_extrapolation(x, y, x_query):
    valid = np.isfinite(x) & np.isfinite(y)
    x_known = x[valid]
    y_known = y[valid]
    if len(x_known) == 0:
        return np.full_like(x_query, np.nan, dtype=float)
    if len(x_known) == 1:
        return np.full_like(x_query, y_known[0], dtype=float)
    order = np.argsort(x_known)
    x_known = x_known[order]
    y_known = y_known[order]
    pred = np.interp(x_query, x_known, y_known)
    left = x_query < x_known[0]
    if left.any():
        slope = (y_known[1] - y_known[0]) / (x_known[1] - x_known[0])
        pred[left] = y_known[0] + slope * (x_query[left] - x_known[0])
    right = x_query > x_known[-1]
    if right.any():
        slope = (y_known[-1] - y_known[-2]) / (x_known[-1] - x_known[-2])
        pred[right] = y_known[-1] + slope * (x_query[right] - x_known[-1])
    return pred


def slope_last(values, x, n):
    known = values.notna()
    if known.sum() < 2:
        return 0.0
    y = values.loc[known].tail(n).to_numpy(dtype=float)
    x_tail = x.loc[known].tail(n).to_numpy(dtype=float)
    if len(y) < 2 or np.isclose(x_tail[-1], x_tail[0]):
        return 0.0
    return float((y[-1] - y[0]) / (x_tail[-1] - x_tail[0]))


def nearest_typewell_by_gr_window(typewell, gr_windows, batch_size=200):
    """Template matching: find the typewell position whose 5-point GR window
    best matches the query window (MSE). Returns matched TVT, MSE, and central GR."""
    type_gr = typewell['GR'].to_numpy(dtype=float)
    type_tvt = typewell['TVT'].to_numpy(dtype=float)
    valid_type = np.isfinite(type_gr)
    type_gr = type_gr[valid_type]
    type_tvt = type_tvt[valid_type]
    m = len(type_gr)
    half = 2
    tw_centers = np.arange(half, m - half)
    tw_windows = np.stack([type_gr[j - half: j + half + 1] for j in tw_centers])
    tw_tvt_centers = type_tvt[tw_centers]
    tw_gr_centers = type_gr[tw_centers]
    n = len(gr_windows)
    matched_tvt = np.empty(n, dtype=float)
    matched_mse = np.empty(n, dtype=float)
    matched_gr = np.empty(n, dtype=float)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = gr_windows[start:end]
        if not np.all(np.isfinite(batch)):
            for k in range(end - start):
                w = batch[k]
                if not np.all(np.isfinite(w)) or m == 0:
                    matched_tvt[start + k] = np.nan
                    matched_mse[start + k] = np.nan
                    matched_gr[start + k] = np.nan
                else:
                    mse = np.mean((tw_windows - w) ** 2, axis=1)
                    best = int(np.argmin(mse))
                    matched_tvt[start + k] = tw_tvt_centers[best]
                    matched_mse[start + k] = float(mse[best])
                    matched_gr[start + k] = tw_gr_centers[best]
        else:
            diff = batch[:, None, :] - tw_windows[None, :, :]
            mse = (diff ** 2).mean(axis=-1)
            best = mse.argmin(axis=1)
            matched_tvt[start:end] = tw_tvt_centers[best]
            matched_mse[start:end] = mse[np.arange(end - start), best]
            matched_gr[start:end] = tw_gr_centers[best]
    return matched_tvt, matched_mse, matched_gr


def build_well_features(horizontal_path, typewell_path, split):
    well_id = horizontal_path.name.split('__')[0]
    horizontal = pd.read_csv(horizontal_path)
    typewell = pd.read_csv(typewell_path)
    row_index = np.arange(len(horizontal))
    tvt_input_missing = horizontal['TVT_input'].isna()
    ps_row = int(tvt_input_missing.idxmax()) if tvt_input_missing.any() else len(horizontal)
    ps_row_safe = min(ps_row, len(horizontal) - 1)
    gr_raw = horizontal['GR']
    gr = gr_raw.interpolate(limit_direction='both')
    md = horizontal['MD']
    baseline_tvt = interpolate_with_extrapolation(
        md.to_numpy(dtype=float),
        horizontal['TVT_input'].to_numpy(dtype=float),
        md.to_numpy(dtype=float),
    )
    typewell_sorted = typewell.sort_values('TVT')
    typewell_gr_at_baseline = interpolate_with_extrapolation(
        typewell_sorted['TVT'].to_numpy(dtype=float),
        typewell_sorted['GR'].to_numpy(dtype=float),
        baseline_tvt,
    )
    known_tvt = horizontal['TVT_input'].dropna()
    first_tvt = float(known_tvt.iloc[0]) if len(known_tvt) else np.nan
    last_tvt = float(known_tvt.iloc[-1]) if len(known_tvt) else np.nan
    gr_first = float(gr.iloc[0])
    gr_last = float(gr.iloc[-1])
    gr_lag_1 = gr.shift(1).fillna(gr_first)
    gr_lag_2 = gr.shift(2).fillna(gr_first)
    gr_lag_3 = gr.shift(3).fillna(gr_first)
    gr_lag_4 = gr.shift(4).fillna(gr_first)
    gr_lag_5 = gr.shift(5).fillna(gr_first)
    gr_lead_1 = gr.shift(-1).fillna(gr_last)
    gr_lead_2 = gr.shift(-2).fillna(gr_last)
    gr_lead_3 = gr.shift(-3).fillna(gr_last)
    gr_lead_4 = gr.shift(-4).fillna(gr_last)
    gr_lead_5 = gr.shift(-5).fillna(gr_last)
    gr_local_window = pd.concat([gr_lag_2, gr_lag_1, gr, gr_lead_1, gr_lead_2], axis=1)
    gr_local_mean_5 = gr_local_window.mean(axis=1)
    gr_local_std_5 = gr_local_window.std(axis=1).fillna(0.0)
    gr_anomaly = gr - gr.rolling(51, center=True, min_periods=1).mean()
    gr_window_matrix = gr_local_window.to_numpy(dtype=float)
    nearest_typewell_tvt_window, nearest_typewell_window_mse, typewell_gr_at_window_match = (
        nearest_typewell_by_gr_window(typewell, gr_window_matrix)
    )
    typewell_tvt_vs_baseline = nearest_typewell_tvt_window - baseline_tvt
    gr_minus_typewell_gr_at_window_match = gr.to_numpy(dtype=float) - typewell_gr_at_window_match
    features = pd.DataFrame({
        'split': split,
        'well_id': well_id,
        'row_index': row_index,
        'n_rows': len(horizontal),
        'row_from_ps': row_index - ps_row,
        'frac_from_ps': (row_index - ps_row) / max(len(horizontal) - ps_row, 1),
        'MD': md,
        'md_from_ps': md - float(md.iloc[ps_row_safe]),
        'Z': horizontal['Z'],
        'x_from_ps': horizontal['X'] - float(horizontal['X'].iloc[ps_row_safe]),
        'y_from_ps': horizontal['Y'] - float(horizontal['Y'].iloc[ps_row_safe]),
        'z_from_ps': horizontal['Z'] - float(horizontal['Z'].iloc[ps_row_safe]),
        'gr': gr,
        'gr_was_missing': gr_raw.isna().astype(int),
        'gr_from_ps': gr - float(gr.iloc[ps_row_safe]),
        'gr_roll_mean_11': gr.rolling(11, center=True, min_periods=1).mean(),
        'gr_roll_mean_51': gr.rolling(51, center=True, min_periods=1).mean(),
        'gr_roll_std_51': gr.rolling(51, center=True, min_periods=2).std().fillna(0.0),
        'gr_delta_1': gr.diff(1).fillna(0.0),
        'gr_delta_10': gr.diff(10).fillna(0.0),
        'last_tvt_input': last_tvt,
        'first_tvt_input': first_tvt,
        'tvt_input_range': last_tvt - first_tvt,
        'tvt_slope_last_25': slope_last(horizontal['TVT_input'], md, 25),
        'tvt_slope_last_100': slope_last(horizontal['TVT_input'], md, 100),
        'baseline_tvt': baseline_tvt,
        'typewell_tvt_min': typewell['TVT'].min(),
        'typewell_tvt_max': typewell['TVT'].max(),
        'typewell_gr_at_baseline_tvt': typewell_gr_at_baseline,
        'gr_minus_typewell_gr_at_baseline': gr.to_numpy(dtype=float) - typewell_gr_at_baseline,
        'gr_lag_1': gr_lag_1, 'gr_lag_2': gr_lag_2, 'gr_lag_3': gr_lag_3,
        'gr_lag_4': gr_lag_4, 'gr_lag_5': gr_lag_5,
        'gr_lead_1': gr_lead_1, 'gr_lead_2': gr_lead_2, 'gr_lead_3': gr_lead_3,
        'gr_lead_4': gr_lead_4, 'gr_lead_5': gr_lead_5,
        'gr_local_mean_5': gr_local_mean_5,
        'gr_local_std_5': gr_local_std_5,
        'gr_anomaly': gr_anomaly,
        'nearest_typewell_tvt_by_gr_window': nearest_typewell_tvt_window,
        'nearest_typewell_window_mse': nearest_typewell_window_mse,
        'typewell_gr_at_window_match': typewell_gr_at_window_match,
        'gr_minus_typewell_gr_at_window_match': gr_minus_typewell_gr_at_window_match,
        'typewell_tvt_vs_baseline': typewell_tvt_vs_baseline,
    })
    features['xy_dist_from_ps'] = np.sqrt(features['x_from_ps'] ** 2 + features['y_from_ps'] ** 2)
    features['is_prediction_row'] = tvt_input_missing.astype(int)
    if 'TVT' in horizontal.columns:
        features['target_tvt'] = horizontal['TVT']
    return features


def build_split(raw_dir, split, exclude_wells=None):
    frames = []
    for horizontal_path in sorted((raw_dir / split).glob('*__horizontal_well.csv')):
        well_id = horizontal_path.name.split('__')[0]
        if exclude_wells and well_id in exclude_wells:
            continue
        typewell_path = raw_dir / split / f'{well_id}__typewell.csv'
        frames.append(build_well_features(horizontal_path, typewell_path, split))
    return pd.concat(frames, ignore_index=True)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [ ]:
# ── Build features ─────────────────────────────────────────────────────────────
print('Building train features (excluding test wells)...')
train_all = build_split(RAW_DIR, 'train', exclude_wells=TEST_WELL_IDS)
train_features = train_all[train_all['is_prediction_row'] == 1].copy()
print(f'Train prediction rows: {len(train_features):,} from {train_all["well_id"].nunique()} wells')

print('Building held-out features for the 3 test wells (from raw/train with full TVT)...')
heldout_frames = []
for wid in sorted(TEST_WELL_IDS):
    h = RAW_DIR / 'train' / f'{wid}__horizontal_well.csv'
    t = RAW_DIR / 'train' / f'{wid}__typewell.csv'
    heldout_frames.append(build_well_features(h, t, 'train'))
heldout_all = pd.concat(heldout_frames, ignore_index=True)
heldout_features = heldout_all[heldout_all['is_prediction_row'] == 1].copy()
print(f'Held-out rows: {len(heldout_features):,}')

print('Building test features (raw/test, for submission)...')
test_all = build_split(RAW_DIR, 'test')
test_features = test_all[test_all['is_prediction_row'] == 1].copy()
print(f'Test prediction rows: {len(test_features):,}')

In [ ]:
# ── Train on 100% of clean train data ─────────────────────────────────────────
print(f'Training LightGBM on {len(train_features):,} rows ({train_features["well_id"].nunique()} wells)...')

model = LGBMRegressor(
    objective='regression',
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=0.05,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbosity=1,
)
model.fit(train_features[FEATURE_COLUMNS], train_features[TARGET_COLUMN])
print('Training complete.')

In [ ]:
# ── Evaluate on held-out test wells (proxy for Kaggle score) ──────────────────
heldout_pred = model.predict(heldout_features[FEATURE_COLUMNS])
heldout_true = heldout_features[TARGET_COLUMN].to_numpy(dtype=float)
heldout_baseline = heldout_features['baseline_tvt'].to_numpy(dtype=float)

print(f'Held-out RMSE (3 test wells, proxy for Kaggle): {rmse(heldout_true, heldout_pred):.4f}')
print(f'Held-out baseline RMSE:                         {rmse(heldout_true, heldout_baseline):.4f}')
print()
for wid in sorted(TEST_WELL_IDS):
    mask = heldout_features['well_id'] == wid
    y_true = heldout_features.loc[mask, TARGET_COLUMN].to_numpy(dtype=float)
    y_pred = heldout_pred[mask.to_numpy()]
    print(f'  {wid}: RMSE={rmse(y_true, y_pred):.4f} ({mask.sum()} rows)')

In [ ]:
# ── Generate submission ────────────────────────────────────────────────────────
test_pred = model.predict(test_features[FEATURE_COLUMNS])
prediction = pd.DataFrame({
    'id': test_features['well_id'].astype(str) + '_' + test_features['row_index'].astype(str),
    'tvt': test_pred,
})
sample = pd.read_csv(RAW_DIR / 'sample_submission.csv')
submission = sample[['id']].merge(prediction, on='id', how='left')
assert not submission['tvt'].isna().any(), 'Missing predictions in submission!'
submission.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote submission to {OUTPUT_PATH}')
print(submission.head())